# Test Notebook for AI Inspection Report Automation

This notebook demonstrates how to test the system's functionalities directly and launch the Streamlit app.

The data folder is set to `../data` relative to the notebook location.

In [1]:
# Ensure the project root is in sys.path
import sys, os
ROOT = os.path.abspath(os.path.join('..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Set the data folder path
DATA_FOLDER = os.path.abspath(os.path.join('..', 'data'))
print(f'Data folder: {DATA_FOLDER}')

Data folder: d:\Projects\data


In [2]:
# Hardcoded paths for testing
scanned_path = os.path.join(DATA_FOLDER, 'tax_details.pdf')
handwritten_path = os.path.join(DATA_FOLDER, 'PCA.pdf')
image_path = os.path.join(DATA_FOLDER, 'images')
print(f'Scanned PDF: {scanned_path}')
print(f'Handwritten PDF: {handwritten_path}')
print(f'Image folder: {image_path}')

Scanned PDF: d:\Projects\data\tax_details.pdf
Handwritten PDF: d:\Projects\data\PCA.pdf
Image folder: d:\Projects\data\images


In [ ]:
import sys
sys.path.append('/content/AI-Report-Automation')  # Adjust the path as needed

In [3]:
# Create project and process PDFs
from pathlib import Path
import sys

# project root
ROOT = Path('inspection_ai/app.py').resolve().parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


import asyncio

from inspection_ai.services import ProjectManager, PDFProcessor

pm = ProjectManager()
pm.create_project("Test Project from Notebook")
print(f"Project created: {pm.project_id}")


async def process_pdfs():
    pdf_processor = PDFProcessor(pm)
    return await pdf_processor.process_pdfs(handwritten_path, scanned_path)


pdf_results = asyncio.run(process_pdfs())
print(f"PDF processing complete: {list(pdf_results.keys())}")

KeyboardInterrupt: 

In [ ]:
# Process all images (120 images)
import glob

from inspection_ai.services import ImageProcessor

image_files = glob.glob(os.path.join(image_path, '*.jpg'))
print(f'Found {len(image_files)} images')

async def process_images():
    image_processor = ImageProcessor(pm)
    return await image_processor.process_images(image_files)

image_results = asyncio.run(process_images())
print(f'Image processing complete: {len(image_results)} images analyzed')

In [ ]:
# Transform and display results
from inspection_ai.services import transform_extracted_data

extracted_data = transform_extracted_data(pdf_results, image_results)
print('Extracted data keys:', list(extracted_data.keys()) if extracted_data else 'No data')

In [ ]:
# Run Playwright extraction for Quire mapping
import extract

print('Running Playwright extraction...')
extract.run()
print('Playwright extraction complete. Check output.json for results.')

In [ ]:
# Map extracted data to Quire fields
import json

from inspection_ai.services import MappingService

# Load Quire fields from output.json
with open('output.json', 'r') as f:
    quire_data = json.load(f)

quire_fields = quire_data.get('all_fields_flat', [])
print(f'Loaded {len(quire_fields)} Quire fields')

# Create mapping service and map fields
mapping_service = MappingService(pm)
mapping_result = mapping_service.map_fields(extracted_data, quire_fields, user_approved=True)

print(f'Mapped {len(mapping_result.get("field_mappings", []))} fields')
print(f'Unmapped fields: {mapping_result.get("unmapped_fields", [])[:5]}...')

In [ ]:
# Launch Streamlit app in a subprocess
import subprocess

app_path = os.path.join(ROOT, 'inspection_ai', 'app.py')
print(f'Starting Streamlit app from: {app_path}')

# Start streamlit in a subprocess (non-blocking)
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', app_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('Streamlit app is starting...')
print('Open your browser to http://localhost:8501 to view the app.')
print('To stop the app, run: streamlit_process.terminate()')